# SprayLine Database ER Model 說明

這份 notebook 用容易理解的方式說明 SprayLine / Digital Twin 專案的資料庫設計。就算沒有學過資料庫，也可以把它想成一個整理工廠資料的「資料櫃」。

資料庫要回答的問題包括：

- 這一批產品是什麼時候生產的？品質結果如何？
- 噴塗過程中的壓力、流量、厚度、噴幅是否正常？
- filter、nozzle、process state 的診斷結果是什麼？
- UI dashboard 要顯示哪些最新狀態？
- ML 預測與 Omniverse 軌跡最佳化結果要如何和產品資料連起來？

本文件的重點不是要求你背資料庫術語，而是看懂每張表格在存什麼、彼此怎麼連起來，以及 UI / AI / Digital Twin 要怎麼查資料。

## 1. 先用生活化方式理解資料庫

可以把資料庫想成 Excel 的加強版：

| 資料庫概念 | 可以想成 | 在本專案中的例子 |
|---|---|---|
| Table | 一張 Excel 工作表 | `part_history`, `runtime_window`, `diagnosis_result` |
| Row | 工作表中的一列資料 | 一個產品、一個感測時間窗、一筆診斷結果 |
| Column | 工作表中的欄位 | `part_uuid`, `start_time`, `pressure_bar` |
| Primary Key | 每列資料的唯一編號 | `part_uuid`, `window_id`, `diagnosis_id` |
| Foreign Key | 指向另一張表的編號 | `runtime_window.part_uuid` 指向 `part_history.part_uuid` |
| Relationship | 表格之間的連線 | 一個 batch 可以包含很多 parts |

ER Model 則是這些表格與連線的地圖。看 ER Model 時，先不用急著看欄位細節，可以先看「哪張表是主角」以及「資料從哪裡接到哪裡」。

## 2. ER Model 圖

下面這張圖是本專案使用的 ER Model。圖中的每個方框是一張資料表，線條表示表格之間的關係。

![Digital Twin ER Model](digital_twin_er_model.svg)

如果在 repository 根目錄的 Markdown 檢視器中開啟，圖片檔案位於：`Database/digital_twin_er_model.svg`。因為這份 notebook 本身也在 `Database` 資料夾內，所以 notebook 內使用相對路徑 `digital_twin_er_model.svg`。

## 3. 這個資料庫分成哪些區塊？

整體設計可以分成六個區塊：

| 區塊 | 目的 | 主要資料表 |
|---|---|---|
| 設定資料 | 儲存配方、零件、產線設定 | `recipe_master`, `component_master`, `production_line_config` |
| 生產紀錄 | 紀錄 batch、part、QC 結果 | `batch_run`, `part_history`, `qc_result` |
| 即時感測資料 | 儲存噴塗過程的壓力、流量、姿態等資料 | `runtime_window`, `runtime_signal`, `runtime_reference`, `runtime_metric`, `robot_pose`, `arm_telemetry_raw` |
| 知識與診斷 | 儲存 threshold 與推論結果 | `filter_threshold`, `nozzle_threshold`, `process_threshold`, `diagnosis_result` |
| ML 預測 | 儲存模型輸入、預測結果、推論紀錄 | `ml_feature_snapshot`, `ml_prediction_result`, `ml_inference_log` |
| Omniverse / Alert | 儲存模擬同步、軌跡最佳化、警示 | `omniverse_sync_log`, `trajectory_optimization_result`, `alert_log` |

這樣分區的好處是：UI 要查最新狀態、工程師要追問題、ML 要拿特徵、Omniverse 要讀寫結果時，都可以找到對應的資料入口。

## 4. 兩條主要資料線

本專案最重要的是兩條資料線：

### 4.1 產品線：從 batch 到 part

這條線回答「某個產品是怎麼生產出來的，品質如何」。

```text
batch_run
  -> part_history
       -> qc_result
       -> component_health_snapshot
       -> ml_feature_snapshot
            -> ml_prediction_result
            -> trajectory_optimization_result
       -> alert_log
```

### 4.2 Runtime 線：從時間窗到診斷

這條線回答「某一段噴塗過程是否正常，異常原因是什麼」。

```text
runtime_window
  -> runtime_signal
  -> runtime_reference
  -> runtime_metric
  -> robot_pose
  -> diagnosis_result
```

`part_history` 和 `runtime_window` 可以用 `part_uuid` 連起來。也就是說，我們可以從一個產品查到它在噴塗時的感測資料與診斷結果。

## 5. 主要資料表說明

下面用「這張表存什麼」的角度說明，不先強調資料庫語法。

### 5.1 設定資料

| Table | 用途 | 重要欄位 |
|---|---|---|
| `recipe_master` | 儲存噴塗配方，例如目標厚度、標準壓力、QC 門檻 | `recipe_id`, `recipe_code`, `target_thickness_mm`, `nozzle_pressure_nom` |
| `component_master` | 儲存設備零件，例如 nozzle、filter、spray gun | `component_id`, `component_type`, `serial_no`, `expected_lifetime_hours`, `status` |
| `production_line_config` | 儲存產線設定值 | `config_key`, `config_value`, `updated_at` |

這些表格比較像系統設定，不會像感測資料一樣每秒大量增加。

### 5.2 生產與品質資料

| Table | 用途 | 重要欄位 |
|---|---|---|
| `batch_run` | 一次生產批次 | `batch_id`, `batch_no`, `recipe_id`, `start_time`, `end_time`, `status` |
| `part_history` | 單一產品的生產紀錄 | `part_uuid`, `batch_id`, `part_no`, `cycle_time_ms`, `total_qc_score`, `status` |
| `qc_result` | 產品的品質檢查結果 | `qc_id`, `part_uuid`, `qc_score`, `thickness_mm`, `defect_count`, `pass_fail` |
| `component_health_snapshot` | 某個產品生產時，零件健康狀態的快照 | `component_id`, `part_uuid`, `clogging_score`, `estimated_remaining_life_h` |

例子：如果 dashboard 顯示某個 part 的 QC 分數偏低，可以從 `part_history` 找到產品，再接到 `qc_result` 看檢測細節，也可以接到 `component_health_snapshot` 看當時 nozzle 或 filter 是否異常。

### 5.3 Runtime / 感測資料

| Table | 用途 | 重要欄位 |
|---|---|---|
| `runtime_window` | 一小段時間的噴塗資料索引 | `window_id`, `line_id`, `batch_id`, `segment_id`, `part_uuid`, `start_time`, `end_time` |
| `runtime_signal` | 實際感測值 | `pressure_bar`, `flow_ml_min`, `spray_width_mm`, `thickness_um`, `temperature_c` |
| `runtime_reference` | 理想或標準值 | `nominal_pressure_bar`, `nominal_flow_ml_min`, `target_thickness_um` |
| `runtime_metric` | 實際值和標準值的差異 | `pressure_error_ratio`, `flow_loss_ratio`, `spray_width_error_ratio`, `thickness_error_ratio` |
| `robot_pose` | 機械手臂位置與速度 | `tcp_x_mm`, `tcp_y_mm`, `tcp_z_mm`, `speed_mm_s`, `stand_off_mm` |
| `arm_telemetry_raw` | 原始 telemetry，用於保留較細的歷史紀錄 | `telemetry_id`, `part_uuid`, `arm_id`, `sim_timestamp`, `wall_timestamp` |

`runtime_window` 是 runtime 資料的主入口。它像是一個資料夾，裡面可以找到同一段時間的 signal、reference、metric、robot pose 和 diagnosis。

### 5.4 知識與診斷資料

| Table | 用途 | 對應來源 |
|---|---|---|
| `filter_threshold` | 判斷 filter 是否堵塞的門檻 | `SprayLine_3/knowledge/filter_thresholds.csv` |
| `nozzle_threshold` | 判斷 nozzle 是否堵塞或異常的門檻 | `SprayLine_3/knowledge/nozzle_thresholds.csv` |
| `process_threshold` | 判斷製程是否穩定的門檻 | `SprayLine_3/knowledge/process_thresholds.csv` |
| `diagnosis_result` | SPARQL / ontology 或 Python rule 推論出的診斷結果 | `SprayLine_3/output/SprayLine_runtime_inferred_sparql.ttl` |

`diagnosis_result.category` 建議使用三大類：

```text
filter_condition
nozzle_condition
process_state
```

這樣 UI 可以固定用三個區塊顯示：filter 狀態、nozzle 狀態、process 狀態。

### 5.5 ML / Omniverse / Alert 資料

| Table | 用途 | 重要欄位 |
|---|---|---|
| `ml_feature_snapshot` | 儲存送進 ML 模型的輸入特徵 | `feature_id`, `part_uuid`, `feature_version`, `arm_features`, `cross_arm_features` |
| `ml_prediction_result` | 儲存 ML 預測結果，例如 QC 或 RUL | `prediction_id`, `prediction_type`, `confidence_score`, `model_version` |
| `ml_inference_log` | 紀錄模型推論是否成功、耗時多久 | `inference_id`, `latency_ms`, `status`, `error_message` |
| `omniverse_sync_log` | 紀錄本地系統和 Omniverse 的同步請求 | `sync_id`, `direction`, `request_type`, `status`, `endpoint_url` |
| `trajectory_optimization_result` | 儲存軌跡最佳化結果 | `optimized_trajectory`, `estimated_qc_gain`, `collision_free`, `validation_status` |
| `alert_log` | 儲存警示與處理狀態 | `alert_type`, `severity`, `message`, `triggered_at`, `acknowledged_at`, `resolved_at` |

這些表格讓系統不只記錄過去發生什麼，也能保存 AI 預測、模擬驗證和警示處理紀錄。

## 6. 表格之間怎麼連？

如果沒有資料庫背景，可以先把 1:N 看成「一個對多個」。例如一個 batch 可以生產很多 parts。

| From | To | 關係 | 白話說明 |
|---|---|---|---|
| `recipe_master` | `batch_run` | 1:N | 一個配方可以被很多批次使用 |
| `batch_run` | `part_history` | 1:N | 一個批次會生產很多產品 |
| `part_history` | `qc_result` | 1:1 | 一個產品通常有一份 QC 結果 |
| `part_history` | `runtime_window` | 1:N | 一個產品可以對應多段 runtime window |
| `runtime_window` | `runtime_signal` | 1:1 | 一段時間窗有一份實際感測值 |
| `runtime_window` | `runtime_reference` | 1:1 | 一段時間窗有一份標準值 |
| `runtime_window` | `runtime_metric` | 1:1 | 一段時間窗有一份偏差計算結果 |
| `runtime_window` | `diagnosis_result` | 1:N | 一段時間窗可以有 filter、nozzle、process 多種診斷 |
| `part_history` | `ml_feature_snapshot` | 1:N | 一個產品可以有不同版本的 ML 特徵 |
| `ml_feature_snapshot` | `ml_prediction_result` | 1:N | 一份特徵可以產生多種預測 |
| `ml_prediction_result` | `alert_log` | 1:N | 一個預測結果可能觸發多個警示 |

`part_uuid` 是產品線和 runtime 線最重要的連接點。只要有 `part_uuid`，就能從產品資料追到感測資料、診斷結果、品質結果和 ML 預測。

## 7. Runtime Metric 怎麼算？

Runtime metric 是把「實際值」和「標準值」做比較，讓系統可以用數字判斷異常程度。

```text
pressure_error_ratio = abs(pressure_bar - nominal_pressure_bar) / nominal_pressure_bar
flow_loss_ratio = max(0, nominal_flow_ml_min - flow_ml_min) / nominal_flow_ml_min
spray_width_error_ratio = abs(spray_width_mm - nominal_spray_width_mm) / nominal_spray_width_mm
thickness_error_ratio = abs(thickness_um - target_thickness_um) / target_thickness_um
```

白話例子：如果標準壓力是 10 bar，實際壓力是 8 bar，壓力誤差比例就是 `abs(8 - 10) / 10 = 0.2`，也就是 20%。這個數字可以再拿去和 threshold 比較，判斷是否異常。

In [ ]:
-- SQL schema example only. This cell is documentation and should not be executed here.

CREATE EXTENSION IF NOT EXISTS timescaledb;
CREATE EXTENSION IF NOT EXISTS pgcrypto;

CREATE TABLE batch_run (
    batch_id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    batch_no VARCHAR(64) NOT NULL,
    recipe_id INT,
    start_time TIMESTAMPTZ NOT NULL,
    end_time TIMESTAMPTZ,
    part_count INT DEFAULT 0,
    status VARCHAR(32) NOT NULL,
    operator_id VARCHAR(64)
);

CREATE TABLE part_history (
    part_uuid UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    batch_id UUID REFERENCES batch_run(batch_id),
    part_no INT,
    recipe_id INT,
    cycle_time_ms INT,
    total_qc_score REAL,
    coating_thickness_mm REAL,
    status VARCHAR(32),
    created_at TIMESTAMPTZ DEFAULT now()
);

CREATE TABLE runtime_window (
    window_id VARCHAR(64) PRIMARY KEY,
    line_id VARCHAR(64) NOT NULL,
    batch_id VARCHAR(64) NOT NULL,
    segment_id VARCHAR(64) NOT NULL,
    part_uuid UUID NULL REFERENCES part_history(part_uuid),
    start_time TIMESTAMPTZ NOT NULL,
    end_time TIMESTAMPTZ NOT NULL,
    source_file TEXT,
    created_at TIMESTAMPTZ DEFAULT now()
);

CREATE TABLE runtime_metric (
    window_id VARCHAR(64) PRIMARY KEY REFERENCES runtime_window(window_id),
    pressure_error_ratio DOUBLE PRECISION NOT NULL,
    flow_loss_ratio DOUBLE PRECISION NOT NULL,
    spray_width_error_ratio DOUBLE PRECISION NOT NULL,
    thickness_error_ratio DOUBLE PRECISION NOT NULL
);

CREATE TABLE diagnosis_result (
    diagnosis_id UUID PRIMARY KEY DEFAULT gen_random_uuid(),
    window_id VARCHAR(64) REFERENCES runtime_window(window_id),
    category VARCHAR(32) NOT NULL,
    state_id VARCHAR(64) NOT NULL,
    state_label VARCHAR(64),
    severity_rank INT,
    confidence DOUBLE PRECISION,
    source VARCHAR(32) DEFAULT 'sparql',
    created_at TIMESTAMPTZ DEFAULT now()
);

-- Example Timescale conversion. Execute only in a real TimescaleDB environment.
-- SELECT create_hypertable('runtime_window', 'start_time', if_not_exists => TRUE);


## 8. Query Function 設計

Query function 是 UI 或 API 查資料時會呼叫的函式。它的目的不是讓前端直接碰 SQL，而是把常用查詢包成清楚的函式名稱。

建議放在：

```text
Database/query_functions.py
```

例如 dashboard 想顯示最新狀態，不需要知道資料存在幾張表，只要呼叫 `get_station_dashboard_status(line_id)`。

### 8.1 Import Functions

| Function | 用途 |
|---|---|
| `import_runtime_json(path_or_dir)` | 匯入 SprayLine runtime JSON，寫入 runtime window / signal / reference / metric / robot pose |
| `import_threshold_csvs(knowledge_dir)` | 匯入 filter / nozzle / process threshold CSV |
| `import_inferred_ttl(ttl_path)` | 匯入 SPARQL 推論結果，寫入 `diagnosis_result` |
| `insert_part_history(payload)` | 新增產品生產紀錄 |
| `insert_qc_result(payload)` | 新增 QC 檢測結果 |

匯入函式建議做到 idempotent，也就是同一份資料重複匯入時，不應該產生重複資料。

### 8.2 Dashboard / Diagnosis / ML Query Functions

| Function | 用途 |
|---|---|
| `get_latest_runtime_window(line_id)` | 查某條產線最新 runtime window |
| `get_runtime_window(window_id)` | 查一段 window 的 signal、reference、metric、diagnosis |
| `get_metric_trend(line_id, metric_name, start_time, end_time)` | 查某個 metric 的時間趨勢 |
| `get_latest_diagnosis(line_id)` | 查最新 filter / nozzle / process 診斷 |
| `get_diagnosis_timeline(line_id, category, start_time, end_time)` | 查某種診斷類別的歷史變化 |
| `get_station_dashboard_status(line_id)` | 回傳 dashboard 首頁需要的摘要狀態 |
| `get_component_health_summary(line_id)` | 查 nozzle / filter / spray gun 的健康摘要 |
| `get_recent_quality_trend(line_id)` | 查近期 QC 趨勢 |
| `get_ml_feature_input(part_uuid)` | 組出 ML 模型需要的輸入資料 |
| `get_latest_prediction(part_uuid)` | 查某個 part 的最新 ML 預測 |
| `get_omniverse_validation_result(part_uuid)` | 查 Omniverse collision check / trajectory validation 結果 |

這些函式的回傳格式建議維持簡單，例如 Python 的 `dict` / `list[dict]`，方便 REST API 和 HTML dashboard 使用。

In [ ]:
# Python query function skeleton only. This cell is documentation and should not be executed here.

from __future__ import annotations

from pathlib import Path
from typing import Any


def import_runtime_json(path_or_dir: str | Path) -> None:
    """Import SprayLine runtime JSON files into runtime tables."""
    raise NotImplementedError


def import_threshold_csvs(knowledge_dir: str | Path) -> None:
    """Import filter/nozzle/process threshold CSV files into knowledge tables."""
    raise NotImplementedError


def import_inferred_ttl(ttl_path: str | Path) -> None:
    """Import SPARQL inferred diagnosis TTL into diagnosis_result."""
    raise NotImplementedError


def get_latest_runtime_window(line_id: str) -> dict[str, Any]:
    """Return the latest runtime window with signal, reference, and metric."""
    raise NotImplementedError


def get_latest_diagnosis(line_id: str) -> dict[str, Any]:
    """Return latest filter/nozzle/process diagnosis for a production line."""
    raise NotImplementedError


def get_metric_trend(
    line_id: str,
    metric_name: str,
    start_time: str,
    end_time: str,
) -> list[dict[str, Any]]:
    """Return time-series metric trend for dashboard or analysis."""
    raise NotImplementedError


def get_station_dashboard_status(line_id: str) -> dict[str, Any]:
    """Return the compact status object needed by the dashboard UI."""
    raise NotImplementedError


def get_ml_feature_input(part_uuid: str) -> dict[str, Any]:
    """Build the ML input payload for QC/RUL/trajectory prediction."""
    raise NotImplementedError


def get_omniverse_validation_result(part_uuid: str) -> dict[str, Any]:
    """Return latest Omniverse validation and trajectory optimization result."""
    raise NotImplementedError


## 9. 建議實作順序

如果要逐步把資料庫做起來，建議分成三個階段：

### Phase 1: 先完成 runtime / diagnosis

```text
runtime JSON
-> runtime tables
-> metric calculation
-> threshold import
-> diagnosis import
-> dashboard query
```

這一階段先讓 UI 可以看到最新感測狀態與診斷結果。

### Phase 2: 接上 production / QC / ML

```text
part_history
-> qc_result
-> component_health_snapshot
-> ml_feature_snapshot
-> ml_prediction_result
-> alert_log
```

這一階段讓系統能從單一 part 追到品質、設備健康、模型預測與警示。

### Phase 3: 接上 API 與 UI

```text
query functions
-> REST endpoints
-> dashboard / manager / search UI
```

這一階段把資料查詢包成 API，例如 `/api/dashboard/latest`、`/api/search/parts`、`/api/manager/replacement-schedule`。

## 10. 結論

這個 ER Model 的核心想法是：

- 用 `part_history` 管理每個產品的生產主紀錄。
- 用 `runtime_window` 管理每段噴塗過程的感測資料。
- 用 `diagnosis_result` 保存 filter、nozzle、process 的推論結果。
- 用 `ml_feature_snapshot` 和 `ml_prediction_result` 保存 AI 模型輸入與輸出。
- 用 `alert_log` 把異常、預測風險和人工處理狀態記錄下來。

對沒有資料庫背景的人來說，先記住一件事就好：這份設計的目的，是讓每個產品都能追到它的生產批次、感測過程、診斷結果、品質結果和 AI / Omniverse 分析結果。